# Retail Demand Forecasting - Step 1: Data Collection

This notebook implements **Step 1 (Data Collection)** of the standard ML lifecycle.

Collection strategy:
- Primary source: **Neon PostgreSQL** (live transactional extraction)
- Fallback source: local raw CSV in `data/raw/`
- Output: standardized raw dataset for downstream preprocessing and feature engineering

## Required Environment Variables
- `NEON_DATABASE_URL` (preferred) or `DATABASE_URL`
- Optional: `NEON_SOURCE_QUERY` to override the default SQL extraction query

You can store these in a `.env` file at the ML engine root.

In [1]:
from pathlib import Path
import os

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 150)

In [2]:
PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_FALLBACK_PATH = RAW_DIR / "DISSANAYAKA POS DATASET.csv"
NEON_SNAPSHOT_PATH = RAW_DIR / "pos_transactions_neon_snapshot.csv"

ENV_PATH = PROJECT_ROOT / ".env"
if ENV_PATH.exists():
    load_dotenv(ENV_PATH)

DATABASE_URL = os.getenv("NEON_DATABASE_URL") or os.getenv("DATABASE_URL")

DEFAULT_QUERY = """
SELECT
    s.receipt_no AS "TransactionID",
    TO_CHAR(s.sale_date, 'DD/MM/YYYY') AS "Date",
    TO_CHAR(s.sale_date, 'HH24:MI:SS') AS "Time",
    COALESCE(si.product_id::text, p.sku, si.product_name) AS "ProductID",
    COALESCE(p.product_name, si.product_name) AS "ProductName",
    COALESCE(p.category, 'Unknown') AS "Category",
    COALESCE(p.unit, 'Unit') AS "PricingUnit",
    COALESCE(si.unit_price, p.selling_price::double precision, 0) AS "UnitPrice",
    COALESCE(p.buying_price::double precision, 0) AS "BuyingPrice",
    COALESCE(p.selling_price::double precision, si.unit_price, 0) AS "SellingPrice",
    COALESCE(si.quantity::double precision, 0) AS "Quantity",
    COALESCE(s.total_amount, 0) AS "Total_LKR",
    COALESCE(s.payment_method, 'Unknown') AS "Payment Method"
FROM sales s
JOIN sale_items si ON si.sale_id = s.id
LEFT JOIN products p ON p.id = si.product_id
ORDER BY s.sale_date ASC;
"""

QUERY = os.getenv("NEON_SOURCE_QUERY") or DEFAULT_QUERY

print("Project root:", PROJECT_ROOT)
print("Local fallback file exists:", LOCAL_FALLBACK_PATH.exists())
print("Neon URL configured:", bool(DATABASE_URL))

Project root: D:\Project\GitHub Project\Dissanayaka Super Web POS\Dissanayake-Super-Web-POS-ML-Engine
Local fallback file exists: True
Neon URL configured: False


In [3]:
def fetch_from_neon(database_url: str, query: str) -> pd.DataFrame:
    if not database_url:
        raise ValueError("DATABASE_URL is missing.")

    engine = create_engine(database_url, pool_pre_ping=True)
    try:
        with engine.connect() as conn:
            df = pd.read_sql(text(query), conn)
    finally:
        engine.dispose()

    return df


def load_step1_raw_dataset() -> tuple[pd.DataFrame, str]:
    if DATABASE_URL:
        try:
            neon_df = fetch_from_neon(DATABASE_URL, QUERY)
            neon_df.to_csv(NEON_SNAPSHOT_PATH, index=False)
            return neon_df, "neon_db"
        except (SQLAlchemyError, ValueError, Exception) as ex:
            print(f"Neon fetch failed. Falling back to local CSV. Reason: {ex}")

    if LOCAL_FALLBACK_PATH.exists():
        fallback_df = pd.read_csv(LOCAL_FALLBACK_PATH, low_memory=False)
        return fallback_df, "local_csv_fallback"

    raise FileNotFoundError(
        f"No data source available. Configure Neon DB or provide fallback file: {LOCAL_FALLBACK_PATH}"
    )

In [4]:
raw_df, data_source = load_step1_raw_dataset()

required_cols = [
    "Date",
    "Time",
    "ProductID",
    "ProductName",
    "Quantity",
    "Total_LKR",
]

missing_cols = [c for c in required_cols if c not in raw_df.columns]
if missing_cols:
    raise ValueError(f"Dataset is missing required columns: {missing_cols}")

raw_df["Quantity"] = pd.to_numeric(raw_df["Quantity"], errors="coerce")
raw_df["Total_LKR"] = pd.to_numeric(raw_df["Total_LKR"], errors="coerce")
raw_df = raw_df.dropna(subset=["Date", "Time", "ProductID", "ProductName", "Quantity"]).copy()

print(f"Data source: {data_source}")
print(f"Rows: {len(raw_df):,}")
print(f"Unique products: {raw_df['ProductID'].nunique():,}")
raw_df.head()

Data source: local_csv_fallback
Rows: 812,258
Unique products: 3,833


,TransactionID,Date,Time,ProductID,ProductName,Category,PricingUnit,UnitPrice,BuyingPrice,SellingPrice,Quantity,Total_LKR,Payment Method
0,TX0005400000,1/1/2021,7:47:47,PI00469,Whiskas Adult Ocean Fish Food - 500g,Pet Products,Pack,820,700,820,1,970,Cash
1,TX0005400000,1/1/2021,7:47:47,PI00097,Kotmale Milk Chocolate - 170ml,Dairy,Pack,50,45,150,3,970,Cash
2,TX0005400001,1/1/2021,8:08:33,PI03359,Precare Unisex Sandal 1 S07 Bj0003 - 1pc,Health & Beauty,Unit,5500,4600,5500,1,5980,Cash
3,TX0005400001,1/1/2021,8:08:33,PI03268,Newtro Chili Powder - 100g,Seeds & Spices,Pack,190,170,380,2,5980,Cash
4,TX0005400001,1/1/2021,8:08:33,PI00173,Denta Comfort Soft Toothbrushes - 2pcs,Health & Beauty,Unit,100,80,100,1,5980,Cash


In [5]:
STEP1_OUT = RAW_DIR / "step1_collected_dataset.csv"
raw_df.to_csv(STEP1_OUT, index=False)
print(f"Step 1 dataset saved to: {STEP1_OUT.resolve()}")

Step 1 dataset saved to: D:\Project\GitHub Project\Dissanayaka Super Web POS\Dissanayake-Super-Web-POS-ML-Engine\data\raw\step1_collected_dataset.csv


## Step 2 - Data Preprocessing

This section performs production-style preprocessing on the Step 1 collected dataset.

Preprocessing actions:
- Build a reliable timestamp from Date and Time
- Remove invalid and duplicate rows
- Handle missing numeric fields with robust fallback logic
- Clip extreme outliers in Quantity and Total_LKR using IQR bounds
- Save cleaned dataset for feature engineering in Step 3

In [6]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

STEP2_OUT = PROCESSED_DIR / "step2_preprocessed_dataset.csv"


def _iqr_clip_bounds(series: pd.Series, factor: float = 1.5) -> tuple[float, float]:
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1

    if pd.isna(iqr) or iqr == 0:
        return float(series.min()), float(series.max())

    lower = q1 - factor * iqr
    upper = q3 + factor * iqr
    return float(lower), float(upper)


def preprocess_step2(frame: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    work = frame.copy()

    report = {
        "rows_in": int(len(work)),
        "rows_after_timestamp": 0,
        "rows_after_dedup": 0,
        "rows_after_quality": 0,
        "quantity_outliers_clipped": 0,
        "total_lkr_outliers_clipped": 0,
    }

    work["timestamp"] = pd.to_datetime(
        work["Date"].astype(str) + " " + work["Time"].astype(str),
        dayfirst=True,
        errors="coerce",
    )

    work["Quantity"] = pd.to_numeric(work["Quantity"], errors="coerce")
    work["Total_LKR"] = pd.to_numeric(work["Total_LKR"], errors="coerce")

    work = work.dropna(subset=["timestamp", "ProductID", "ProductName", "Quantity"]).copy()
    report["rows_after_timestamp"] = int(len(work))

    dedup_keys = ["timestamp", "ProductID", "ProductName", "Quantity", "Total_LKR"]
    work = work.drop_duplicates(subset=dedup_keys).copy()
    report["rows_after_dedup"] = int(len(work))

    work = work[work["Quantity"] > 0].copy()

    if "Total_LKR" in work.columns:
        work["Total_LKR"] = work["Total_LKR"].fillna(work["Quantity"] * work.get("UnitPrice", 0))
        work["Total_LKR"] = work["Total_LKR"].fillna(0)

    q_low, q_high = _iqr_clip_bounds(work["Quantity"])
    q_before = work["Quantity"].copy()
    work["Quantity"] = work["Quantity"].clip(lower=max(0, q_low), upper=q_high)
    report["quantity_outliers_clipped"] = int((q_before != work["Quantity"]).sum())

    if "Total_LKR" in work.columns and work["Total_LKR"].notna().any():
        t_low, t_high = _iqr_clip_bounds(work["Total_LKR"])
        t_before = work["Total_LKR"].copy()
        work["Total_LKR"] = work["Total_LKR"].clip(lower=max(0, t_low), upper=t_high)
        report["total_lkr_outliers_clipped"] = int((t_before != work["Total_LKR"]).sum())

    work = work.sort_values(["timestamp", "ProductID"]).reset_index(drop=True)
    report["rows_after_quality"] = int(len(work))

    return work, report

In [7]:
step2_df, step2_report = preprocess_step2(raw_df)

print("Step 2 Report")
for key, value in step2_report.items():
    print(f"- {key}: {value:,}" if isinstance(value, int) else f"- {key}: {value}")

step2_df.head()

Step 2 Report
- rows_in: 812,258
- rows_after_timestamp: 812,258
- rows_after_dedup: 812,258
- rows_after_quality: 812,258
- quantity_outliers_clipped: 0
- total_lkr_outliers_clipped: 38,718


,TransactionID,Date,Time,ProductID,ProductName,Category,PricingUnit,UnitPrice,BuyingPrice,SellingPrice,Quantity,Total_LKR,Payment Method,timestamp
0,TX0005400000,1/1/2021,7:47:47,PI00097,Kotmale Milk Chocolate - 170ml,Dairy,Pack,50,45,150,3,970,Cash,2021-01-01 07:47:47
1,TX0005400000,1/1/2021,7:47:47,PI00469,Whiskas Adult Ocean Fish Food - 500g,Pet Products,Pack,820,700,820,1,970,Cash,2021-01-01 07:47:47
2,TX0005400001,1/1/2021,8:08:33,PI00173,Denta Comfort Soft Toothbrushes - 2pcs,Health & Beauty,Unit,100,80,100,1,5980,Cash,2021-01-01 08:08:33
3,TX0005400001,1/1/2021,8:08:33,PI03268,Newtro Chili Powder - 100g,Seeds & Spices,Pack,190,170,380,2,5980,Cash,2021-01-01 08:08:33
4,TX0005400001,1/1/2021,8:08:33,PI03359,Precare Unisex Sandal 1 S07 Bj0003 - 1pc,Health & Beauty,Unit,5500,4600,5500,1,5980,Cash,2021-01-01 08:08:33


In [8]:
step2_df.to_csv(STEP2_OUT, index=False)
print(f"Step 2 preprocessed dataset saved to: {STEP2_OUT.resolve()}")

Step 2 preprocessed dataset saved to: D:\Project\GitHub Project\Dissanayaka Super Web POS\Dissanayake-Super-Web-POS-ML-Engine\data\processed\step2_preprocessed_dataset.csv


## Step 3: Feature Engineering (Lag + Rolling + Seasonality)

In this step, we transform cleaned transactional data into product-level daily features for forecasting:
- Aggregate transactions to daily product demand.
- Create lag features (1, 7, 14, 28 days).
- Create rolling statistics (7, 14, 28 day windows).
- Add calendar and Sri Lankan seasonality indicators.
- Save a model-ready feature dataset for Step 4 time-based splitting.

In [9]:
STEP3_OUT = PROCESSED_DIR / "step3_feature_engineered_dataset.csv"


def add_sri_lanka_seasonality_flags(frame: pd.DataFrame) -> pd.DataFrame:
    data = frame.copy()

    # Practical season proxies for Sri Lanka retail demand.
    data["is_avurudu_season"] = data["month"].isin([4]).astype(int)
    data["is_vesak_season"] = data["month"].isin([5]).astype(int)
    data["is_year_end_festive"] = data["month"].isin([12]).astype(int)
    data["is_tourist_peak"] = data["month"].isin([12, 1, 2, 3]).astype(int)

    return data


def engineer_step3_features(frame: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    data = frame.copy()

    if "timestamp" not in data.columns:
        raise ValueError("Step 2 data must include a 'timestamp' column.")

    data["timestamp"] = pd.to_datetime(data["timestamp"], errors="coerce")
    data = data.dropna(subset=["timestamp", "ProductID", "Quantity"]).copy()

    data["date"] = data["timestamp"].dt.floor("D")

    daily = (
        data.groupby(["ProductID", "ProductName", "date"], as_index=False)
        .agg(
            daily_qty=("Quantity", "sum"),
            daily_sales_lkr=("Total_LKR", "sum"),
            avg_unit_price=("UnitPrice", "mean"),
            txn_count=("TransactionID", "count"),
        )
        .sort_values(["ProductID", "date"])
        .reset_index(drop=True)
    )

    daily["day_of_week"] = daily["date"].dt.dayofweek
    daily["week_of_year"] = daily["date"].dt.isocalendar().week.astype(int)
    daily["month"] = daily["date"].dt.month
    daily["quarter"] = daily["date"].dt.quarter
    daily["is_weekend"] = daily["day_of_week"].isin([5, 6]).astype(int)

    group = daily.groupby("ProductID", group_keys=False)

    for lag in [1, 7, 14, 28]:
        daily[f"lag_qty_{lag}d"] = group["daily_qty"].shift(lag)

    for window in [7, 14, 28]:
        shifted = group["daily_qty"].shift(1)
        daily[f"roll_mean_qty_{window}d"] = (
            shifted.groupby(daily["ProductID"]).rolling(window, min_periods=3).mean().reset_index(level=0, drop=True)
        )
        daily[f"roll_std_qty_{window}d"] = (
            shifted.groupby(daily["ProductID"]).rolling(window, min_periods=3).std().reset_index(level=0, drop=True)
        )

    daily = add_sri_lanka_seasonality_flags(daily)

    rows_before_dropna = len(daily)
    feature_cols = [
        "lag_qty_1d", "lag_qty_7d", "lag_qty_14d", "lag_qty_28d",
        "roll_mean_qty_7d", "roll_mean_qty_14d", "roll_mean_qty_28d",
    ]
    feature_df = daily.dropna(subset=feature_cols).reset_index(drop=True)

    report = {
        "product_day_rows": int(rows_before_dropna),
        "model_ready_rows": int(len(feature_df)),
        "rows_dropped_for_lags": int(rows_before_dropna - len(feature_df)),
        "unique_products": int(feature_df["ProductID"].nunique() if len(feature_df) else 0),
        "date_min": str(feature_df["date"].min()) if len(feature_df) else "NA",
        "date_max": str(feature_df["date"].max()) if len(feature_df) else "NA",
    }

    return feature_df, report

In [10]:
step3_df, step3_report = engineer_step3_features(step2_df)

print("Step 3 Report")
for key, value in step3_report.items():
    print(f"- {key}: {value:,}" if isinstance(value, int) else f"- {key}: {value}")

step3_df.head()

Step 3 Report
- product_day_rows: 760,806
- model_ready_rows: 653,482
- rows_dropped_for_lags: 107,324
- unique_products: 3,833
- date_min: 2021-03-23 00:00:00
- date_max: 2025-12-31 00:00:00


,ProductID,ProductName,date,daily_qty,daily_sales_lkr,avg_unit_price,txn_count,day_of_week,week_of_year,month,quarter,is_weekend,lag_qty_1d,lag_qty_7d,lag_qty_14d,lag_qty_28d,roll_mean_qty_7d,roll_std_qty_7d,roll_mean_qty_14d,roll_std_qty_14d,roll_mean_qty_28d,roll_std_qty_28d,is_avurudu_season,is_vesak_season,is_year_end_festive,is_tourist_peak
0,PI00001,Captain Oats Instant - 500g,2021-08-25,3,12235,1020.0,1,2,34,8,3,0,1.0,1.0,1.0,2.0,2.285714,1.603567,2.142857,1.292412,2.428571,1.596955,0,0,0,0
1,PI00001,Captain Oats Instant - 500g,2021-09-13,4,8870,1070.0,2,0,37,9,3,0,3.0,5.0,3.0,4.0,2.571429,1.511858,2.285714,1.266647,2.464286,1.598197,0,0,0,0
2,PI00001,Captain Oats Instant - 500g,2021-09-29,3,8370,1080.0,1,2,39,9,3,0,4.0,2.0,2.0,2.0,2.428571,1.272418,2.357143,1.336306,2.464286,1.598197,0,0,0,0
3,PI00001,Captain Oats Instant - 500g,2021-10-03,4,12235,1120.0,1,6,39,10,4,1,3.0,2.0,3.0,1.0,2.571429,1.272418,2.428571,1.342460,2.500000,1.598611,0,0,0,0
4,PI00001,Captain Oats Instant - 500g,2021-10-05,1,4910,1140.0,1,1,40,10,4,0,4.0,4.0,1.0,4.0,2.857143,1.345185,2.500000,1.400549,2.607143,1.594883,0,0,0,0


In [11]:
step3_df.to_csv(STEP3_OUT, index=False)
print(f"Step 3 feature dataset saved to: {STEP3_OUT.resolve()}")

Step 3 feature dataset saved to: D:\Project\GitHub Project\Dissanayaka Super Web POS\Dissanayake-Super-Web-POS-ML-Engine\data\processed\step3_feature_engineered_dataset.csv


## Step 4: Time-Based Train/Validation/Test Split

This step performs chronological splitting without shuffling to avoid temporal leakage:
- Train: earliest period
- Validation: middle period
- Test: latest period

The split is done using sorted unique dates from Step 3 feature data.

In [12]:
STEP4_TRAIN_OUT = PROCESSED_DIR / "step4_train.csv"
STEP4_VAL_OUT = PROCESSED_DIR / "step4_validation.csv"
STEP4_TEST_OUT = PROCESSED_DIR / "step4_test.csv"


def split_by_time(frame: pd.DataFrame, train_ratio: float = 0.70, val_ratio: float = 0.15) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, dict]:
    if "date" not in frame.columns:
        raise ValueError("Step 3 data must include a 'date' column.")

    if train_ratio <= 0 or val_ratio <= 0 or (train_ratio + val_ratio) >= 1:
        raise ValueError("Ratios must be positive and train_ratio + val_ratio must be < 1.")

    data = frame.copy()
    data["date"] = pd.to_datetime(data["date"], errors="coerce")
    data = data.dropna(subset=["date"]).sort_values(["date", "ProductID"]).reset_index(drop=True)

    unique_dates = data["date"].sort_values().drop_duplicates().reset_index(drop=True)

    if len(unique_dates) < 10:
        raise ValueError("Not enough unique dates for robust time-based split.")

    train_end_idx = int(len(unique_dates) * train_ratio)
    val_end_idx = int(len(unique_dates) * (train_ratio + val_ratio))

    train_end_idx = max(1, min(train_end_idx, len(unique_dates) - 2))
    val_end_idx = max(train_end_idx + 1, min(val_end_idx, len(unique_dates) - 1))

    train_last_date = unique_dates.iloc[train_end_idx - 1]
    val_last_date = unique_dates.iloc[val_end_idx - 1]

    train_df = data[data["date"] <= train_last_date].copy()
    val_df = data[(data["date"] > train_last_date) & (data["date"] <= val_last_date)].copy()
    test_df = data[data["date"] > val_last_date].copy()

    report = {
        "total_rows": int(len(data)),
        "total_unique_dates": int(len(unique_dates)),
        "train_rows": int(len(train_df)),
        "validation_rows": int(len(val_df)),
        "test_rows": int(len(test_df)),
        "train_start": str(train_df["date"].min()),
        "train_end": str(train_df["date"].max()),
        "validation_start": str(val_df["date"].min()),
        "validation_end": str(val_df["date"].max()),
        "test_start": str(test_df["date"].min()),
        "test_end": str(test_df["date"].max()),
    }

    return train_df, val_df, test_df, report

In [13]:
train_df, val_df, test_df, step4_report = split_by_time(step3_df, train_ratio=0.70, val_ratio=0.15)

print("Step 4 Report")
for key, value in step4_report.items():
    print(f"- {key}: {value:,}" if isinstance(value, int) else f"- {key}: {value}")

print("\nSample rows")
print("Train:")
display(train_df.head(2))
print("Validation:")
display(val_df.head(2))
print("Test:")
display(test_df.head(2))

Step 4 Report
- total_rows: 653,482
- total_unique_dates: 1,739
- train_rows: 435,872
- validation_rows: 108,629
- test_rows: 108,981
- train_start: 2021-03-23 00:00:00
- train_end: 2024-07-27 00:00:00
- validation_start: 2024-07-28 00:00:00
- validation_end: 2025-04-14 00:00:00
- test_start: 2025-04-15 00:00:00
- test_end: 2025-12-31 00:00:00

Sample rows
Train:


,ProductID,ProductName,date,daily_qty,daily_sales_lkr,avg_unit_price,txn_count,day_of_week,week_of_year,month,quarter,is_weekend,lag_qty_1d,lag_qty_7d,lag_qty_14d,lag_qty_28d,roll_mean_qty_7d,roll_std_qty_7d,roll_mean_qty_14d,roll_std_qty_14d,roll_mean_qty_28d,roll_std_qty_28d,is_avurudu_season,is_vesak_season,is_year_end_festive,is_tourist_peak
0,PI01301,Samaayu Gotukola Porridge - 50g,2021-03-23,3,600,90.0,1,1,12,3,1,0,1.0,5.0,3.0,1.0,2.428571,2.149197,2.428571,1.741542,2.142857,1.380131,0,0,0,1
1,PI00099,Kotmale Milk Chocolate - 180ml,2021-03-25,1,820,70.0,1,3,12,3,1,0,1.0,5.0,3.0,3.0,3.571429,2.225395,2.785714,1.888368,2.357143,1.496026,0,0,0,1


Validation:


,ProductID,ProductName,date,daily_qty,daily_sales_lkr,avg_unit_price,txn_count,day_of_week,week_of_year,month,quarter,is_weekend,lag_qty_1d,lag_qty_7d,lag_qty_14d,lag_qty_28d,roll_mean_qty_7d,roll_std_qty_7d,roll_mean_qty_14d,roll_std_qty_14d,roll_mean_qty_28d,roll_std_qty_28d,is_avurudu_season,is_vesak_season,is_year_end_festive,is_tourist_peak
435872,PI00002,Oateo Instant Oats - 400g,2024-07-28,1,10690,560.0,1,6,30,7,3,1,1.0,5.0,4.0,1.0,2.0,1.414214,2.714286,1.683795,2.464286,1.502643,0,0,0,0
435873,PI00004,Nestle Nestum Original - 250g,2024-07-28,1,6360,1080.0,1,6,30,7,3,1,4.0,1.0,1.0,1.0,3.0,1.632993,2.571429,1.342460,2.392857,1.448864,0,0,0,0


Test:


,ProductID,ProductName,date,daily_qty,daily_sales_lkr,avg_unit_price,txn_count,day_of_week,week_of_year,month,quarter,is_weekend,lag_qty_1d,lag_qty_7d,lag_qty_14d,lag_qty_28d,roll_mean_qty_7d,roll_std_qty_7d,roll_mean_qty_14d,roll_std_qty_14d,roll_mean_qty_28d,roll_std_qty_28d,is_avurudu_season,is_vesak_season,is_year_end_festive,is_tourist_peak
544501,PI00002,Oateo Instant Oats - 400g,2025-04-15,4,4450,540.0,1,1,16,4,2,0,4.0,2.0,1.0,5.0,2.142857,1.069045,2.071429,1.141139,2.928571,1.884215,1,0,0,0
544502,PI00015,Home Bake Sandwich Bread - 500g,2025-04-15,4,1920,410.0,1,1,16,4,2,0,4.0,2.0,3.0,2.0,2.285714,1.112697,2.357143,0.928783,2.535714,1.231745,1,0,0,0


In [14]:
train_df.to_csv(STEP4_TRAIN_OUT, index=False)
val_df.to_csv(STEP4_VAL_OUT, index=False)
test_df.to_csv(STEP4_TEST_OUT, index=False)

print(f"Step 4 train dataset saved to: {STEP4_TRAIN_OUT.resolve()}")
print(f"Step 4 validation dataset saved to: {STEP4_VAL_OUT.resolve()}")
print(f"Step 4 test dataset saved to: {STEP4_TEST_OUT.resolve()}")

Step 4 train dataset saved to: D:\Project\GitHub Project\Dissanayaka Super Web POS\Dissanayake-Super-Web-POS-ML-Engine\data\processed\step4_train.csv
Step 4 validation dataset saved to: D:\Project\GitHub Project\Dissanayaka Super Web POS\Dissanayake-Super-Web-POS-ML-Engine\data\processed\step4_validation.csv
Step 4 test dataset saved to: D:\Project\GitHub Project\Dissanayaka Super Web POS\Dissanayake-Super-Web-POS-ML-Engine\data\processed\step4_test.csv


Step 4 complete. Next: Step 5 model training with RandomForest and XGBoost on the train split.

## Step 5: Model Training

This step trains forecasting models using the Step 4 train split:
- RandomForest Regressor
- XGBoost Regressor (with fallback if XGBoost is unavailable)

Artifacts are saved for Step 6 evaluation and comparison.

In [18]:
import json
from datetime import datetime, timezone

import joblib
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBRegressor = None
    XGBOOST_AVAILABLE = False

MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

STEP5_RF_MODEL_OUT = MODEL_DIR / "step5_random_forest.joblib"
STEP5_BOOST_MODEL_OUT = MODEL_DIR / "step5_boosting_model.joblib"
STEP5_META_OUT = MODEL_DIR / "step5_training_metadata.json"

TARGET_COL = "daily_qty"
FEATURE_COLS = [
    "day_of_week", "week_of_year", "month", "quarter", "is_weekend",
    "lag_qty_1d", "lag_qty_7d", "lag_qty_14d", "lag_qty_28d",
    "roll_mean_qty_7d", "roll_std_qty_7d",
    "roll_mean_qty_14d", "roll_std_qty_14d",
    "roll_mean_qty_28d", "roll_std_qty_28d",
    "is_avurudu_season", "is_vesak_season", "is_year_end_festive", "is_tourist_peak",
    "avg_unit_price",
]


def prepare_xy(frame: pd.DataFrame, feature_cols: list[str], target_col: str) -> tuple[pd.DataFrame, pd.Series]:
    missing = [c for c in feature_cols + [target_col] if c not in frame.columns]
    if missing:
        raise ValueError(f"Missing required columns for Step 5: {missing}")

    x = frame[feature_cols].copy()
    x = x.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    y = pd.to_numeric(frame[target_col], errors="coerce")

    valid_mask = (~x.isna().any(axis=1)) & y.notna()
    x = x.loc[valid_mask].reset_index(drop=True)
    y = y.loc[valid_mask].reset_index(drop=True)

    return x, y

In [19]:
x_train, y_train = prepare_xy(train_df, FEATURE_COLS, TARGET_COL)
x_val, y_val = prepare_xy(val_df, FEATURE_COLS, TARGET_COL)

rf_model = RandomForestRegressor(
    n_estimators=120,
    max_depth=14,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1,
)
rf_model.fit(x_train, y_train)

if XGBOOST_AVAILABLE:
    boost_model = XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        n_jobs=-1,
        random_state=42,
    )
    boost_model.fit(x_train, y_train, eval_set=[(x_val, y_val)], verbose=False)
    boost_model_type = "XGBRegressor"
else:
    boost_model = HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_depth=10,
        max_iter=300,
        random_state=42,
    )
    boost_model.fit(x_train, y_train)
    boost_model_type = "HistGradientBoostingRegressor"

step5_report = {
    "xgboost_available": bool(XGBOOST_AVAILABLE),
    "boost_model_type": boost_model_type,
    "features_used": FEATURE_COLS,
    "target": TARGET_COL,
    "train_rows_used": int(len(x_train)),
    "validation_rows_used": int(len(x_val)),
    "train_date_min": str(train_df["date"].min()),
    "train_date_max": str(train_df["date"].max()),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}

print("Step 5 Report")
for key, value in step5_report.items():
    print(f"- {key}: {value}")

Step 5 Report
- xgboost_available: False
- boost_model_type: HistGradientBoostingRegressor
- features_used: ['day_of_week', 'week_of_year', 'month', 'quarter', 'is_weekend', 'lag_qty_1d', 'lag_qty_7d', 'lag_qty_14d', 'lag_qty_28d', 'roll_mean_qty_7d', 'roll_std_qty_7d', 'roll_mean_qty_14d', 'roll_std_qty_14d', 'roll_mean_qty_28d', 'roll_std_qty_28d', 'is_avurudu_season', 'is_vesak_season', 'is_year_end_festive', 'is_tourist_peak', 'avg_unit_price']
- target: daily_qty
- train_rows_used: 435872
- validation_rows_used: 108629
- train_date_min: 2021-03-23 00:00:00
- train_date_max: 2024-07-27 00:00:00
- created_at_utc: 2026-04-04T21:51:39.307209+00:00


In [20]:
joblib.dump(rf_model, STEP5_RF_MODEL_OUT)
joblib.dump(boost_model, STEP5_BOOST_MODEL_OUT)

with STEP5_META_OUT.open("w", encoding="utf-8") as f:
    json.dump(step5_report, f, indent=2)

print(f"Step 5 RandomForest model saved to: {STEP5_RF_MODEL_OUT.resolve()}")
print(f"Step 5 boosting model saved to: {STEP5_BOOST_MODEL_OUT.resolve()}")
print(f"Step 5 metadata saved to: {STEP5_META_OUT.resolve()}")

Step 5 RandomForest model saved to: D:\Project\GitHub Project\Dissanayaka Super Web POS\Dissanayake-Super-Web-POS-ML-Engine\models\step5_random_forest.joblib
Step 5 boosting model saved to: D:\Project\GitHub Project\Dissanayaka Super Web POS\Dissanayake-Super-Web-POS-ML-Engine\models\step5_boosting_model.joblib
Step 5 metadata saved to: D:\Project\GitHub Project\Dissanayaka Super Web POS\Dissanayake-Super-Web-POS-ML-Engine\models\step5_training_metadata.json


## Step 6: Model Evaluation (Validation + Test)

This step evaluates Step 5 models on validation and test splits using:
- RMSE (lower is better)
- MAE (lower is better)

A winner is selected based on test RMSE, then test MAE as tie-breaker.

In [24]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

STEP6_METRICS_OUT = MODEL_DIR / "step6_evaluation_metrics.json"


def _rmse(y_true, y_pred) -> float:
    mse = mean_squared_error(y_true, y_pred)
    return float(np.sqrt(mse))


def evaluate_model(model_name: str, model, x_data: pd.DataFrame, y_data: pd.Series, split: str) -> dict:
    preds = model.predict(x_data)
    return {
        "model": model_name,
        "split": split,
        "rmse": _rmse(y_data, preds),
        "mae": float(mean_absolute_error(y_data, preds)),
        "rows": int(len(y_data)),
    }

In [25]:
x_test, y_test = prepare_xy(test_df, FEATURE_COLS, TARGET_COL)

results = []
results.append(evaluate_model("RandomForestRegressor", rf_model, x_val, y_val, "validation"))
results.append(evaluate_model("RandomForestRegressor", rf_model, x_test, y_test, "test"))
results.append(evaluate_model(step5_report["boost_model_type"], boost_model, x_val, y_val, "validation"))
results.append(evaluate_model(step5_report["boost_model_type"], boost_model, x_test, y_test, "test"))

results_df = pd.DataFrame(results)

test_results = results_df[results_df["split"] == "test"].sort_values(["rmse", "mae"]).reset_index(drop=True)
winner_model = str(test_results.loc[0, "model"])

step6_report = {
    "target": TARGET_COL,
    "feature_count": len(FEATURE_COLS),
    "validation_rows_used": int(len(y_val)),
    "test_rows_used": int(len(y_test)),
    "winner_model": winner_model,
    "results": results,
}

print("Step 6 Results")
display(results_df.sort_values(["split", "rmse", "mae"]).reset_index(drop=True))
print(f"Winner model (test RMSE, then MAE): {winner_model}")

Step 6 Results


,model,split,rmse,mae,rows
0,RandomForestRegressor,test,1.395422,1.109572,108981
1,HistGradientBoostingRegressor,test,1.395806,1.109242,108981
2,RandomForestRegressor,validation,1.394807,1.110178,108629
3,HistGradientBoostingRegressor,validation,1.394900,1.109994,108629


Winner model (test RMSE, then MAE): RandomForestRegressor


In [26]:
with STEP6_METRICS_OUT.open("w", encoding="utf-8") as f:
    json.dump(step6_report, f, indent=2)

print(f"Step 6 evaluation metrics saved to: {STEP6_METRICS_OUT.resolve()}")

Step 6 evaluation metrics saved to: D:\Project\GitHub Project\Dissanayaka Super Web POS\Dissanayake-Super-Web-POS-ML-Engine\models\step6_evaluation_metrics.json


Step 6 complete. Next: Step 7 hyperparameter tuning and cross-validation on shortlisted model(s).